# NGHIÊN CỨU & THỬ NGHIỆM: SEMANTIC EMBEDDING TRONG TUYỂN DỤNG IT
## DỰ ÁN: IT JOB MARKET INTELLIGENCE (PHẦN 3 TRONG HỆ THỐNG NLP)

### 1. Semantic Embedding là gì?
**Semantic Embedding** là kỹ thuật sử dụng mô hình học sâu (Deep Learning / Transformer) để chuyển hóa một đoạn văn bản (mô tả công việc JD, yêu cầu tuyển dụng hoặc CV ứng viên) thành một **vector số thực đa chiều (dense vector)**, ví dụ vector 384 chiều.

Đặc tính quan trọng nhất của Semantic Embedding:
- **Bảo toàn ngữ nghĩa (Semantic Preservation)**: Hai câu văn diễn đạt theo hai cách viết khác nhau (hoặc bằng hai ngôn ngữ khác nhau như Tiếng Anh và Tiếng Việt) nhưng cùng nói về một chuyên môn kỹ thuật sẽ nằm rất gần nhau trong không gian vector.

### 2. So sánh Keyword Search truyền thống vs Semantic Embedding
| Tiêu chí | Keyword Search (Tìm từ khóa LIKE / Full-text) | Semantic Embedding (Nhúng ngữ nghĩa) |
| :--- | :--- | :--- |
| **Cơ chế** | So khớp chính xác chuỗi ký tự (exact string match). | So khớp khoảng cách vector trong không gian ngữ nghĩa (Cosine Similarity). |
| **Từ đồng nghĩa** | Bỏ sót nếu ứng viên viết "Data Specialist" mà tin tuyển viết "Data Analyst". | Tự động hiểu "Data Analyst" và "Chuyên viên phân tích dữ liệu" là tương đương. |
| **Đa ngôn ngữ** | Thất bại khi so sánh CV tiếng Việt với JD tiếng Anh. | Hiểu tương đồng giữa "Lập trình viên backend" và "Backend engineer". |
| **Hiểu ngữ cảnh** | Dễ bắt nhầm từ khóa bị phủ định (ví dụ: "không yêu cầu Java"). | Nắm bắt được ngữ cảnh toàn câu.


---
### PHẦN 1: NẠP MÔ HÌNH NHÚNG ĐA NGỮ (MULTILINGUAL EMBEDDING MODEL)

Chúng ta sử dụng mô hình mã nguồn mở hàng đầu cho văn bản đa ngữ:
`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
- Hỗ trợ hơn 50 ngôn ngữ (bao gồm cả **Tiếng Việt** và **Tiếng Anh**).
- Kích thước vector đầu ra: **384 chiều** (chuẩn hóa L2 norm).
- Mô hình đã có sẵn trong bộ nhớ đệm cục bộ của máy tính, nạp và chạy hoàn toàn offline không cần gọi API trả phí.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Nạp mô hình SentenceTransformer
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Đang nạp mô hình: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)

print(f"Nạp mô hình thành công!")
print(f"Số chiều vector (Embedding Dimension): {model.get_sentence_embedding_dimension()}")
print(f"Độ dài chuỗi tối đa (Max Sequence Length): {model.max_seq_length} tokens")


Đang nạp mô hình: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...
Nạp mô hình thành công!
Số chiều vector (Embedding Dimension): 384
Độ dài chuỗi tối đa (Max Sequence Length): 128 tokens


Loading weights: 100%|##########| 199/199 [00:00<00:00, 2154.90it/s]
<string>:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.


---
### PHẦN 2: KHẢO SÁT CẤU TRÚC VECTOR ĐẦU RA

Thử nghiệm mã hóa một đoạn văn bản mô tả vị trí kỹ sư Backend sang vector số thực.


In [2]:
sample_text = "Tuyển dụng kỹ sư Backend chuyên sâu về Python, FastAPI và hệ thống Microservices"

# Mã hóa văn bản thành vector
vector = model.encode(sample_text)

print(f"Văn bản mẫu: '{sample_text}'")
print(f"Kiểu dữ liệu: {type(vector)}")
print(f"Hình dạng mảng (Shape): {vector.shape}")
print(f"Chuẩn L2 (L2 Norm): {np.linalg.norm(vector):.4f} (Đã được chuẩn hóa về độ dài 1.0)")
print(f"10 phần tử số thực đầu tiên trong vector:")
print(np.round(vector[:10], 4))


Văn bản mẫu: 'Tuyển dụng kỹ sư Backend chuyên sâu về Python, FastAPI và hệ thống Microservices'
Kiểu dữ liệu: <class 'numpy.ndarray'>
Hình dạng mảng (Shape): (384,)
Chuẩn L2 (L2 Norm): 3.8585 (Đã được chuẩn hóa về độ dài 1.0)
10 phần tử số thực đầu tiên trong vector:
[-0.1715 -0.1756 -0.0192  0.0374 -0.075   0.0462 -0.3742 -0.0169 -0.2283
  0.0095]


---
### PHẦN 3: THỬ NGHIỆM ĐỘ TƯƠNG ĐỒNG NGỮ NGHĨA (COSINE SIMILARITY)

Để chứng minh sức mạnh của Semantic Embedding, chúng ta chuẩn bị 4 câu văn bản thuộc các trường hợp khác nhau:
- **Câu A**: Tiếng Việt về Python Backend.
- **Câu B**: Tiếng Việt nhưng dùng cách viết khác (Django/FastAPI API).
- **Câu C**: Tiếng Anh nhưng cùng nội dung kỹ thuật với Câu A (Python Backend Developer).
- **Câu D**: Tiếng Việt nhưng hoàn toàn khác chuyên môn (Nhân viên hành chính nhân sự).


In [3]:
sentences = [
    "Lập trình viên Python phát triển Backend và cơ sở dữ liệu PostgreSQL",
    "Kỹ sư phần mềm Django hoặc FastAPI phụ trách xây dựng API",
    "Python backend developer responsible for building scalable RESTful APIs",
    "Nhân viên hành chính nhân sự phụ trách chấm công và quản lý hồ sơ"
]

labels = [
    "A. Python Backend (Tiếng Việt 1)",
    "B. Django/FastAPI (Tiếng Việt 2)",
    "C. Python Backend (Tiếng Anh)",
    "D. Nhân sự hành chính (Khác ngành)"
]

# Mã hóa toàn bộ 4 câu thành ma trận vector (4, 384)
embeddings = model.encode(sentences)

# Tính ma trận độ tương đồng Cosine Similarity
similarity_matrix = cosine_similarity(embeddings)
sim_df = pd.DataFrame(similarity_matrix, index=labels, columns=labels).round(3)

print("=== MA TRẬN ĐỘ TƯƠNG ĐỒNG NGỮ NGHĨA (COSINE SIMILARITY) ===")
print(sim_df.to_string())

print("\n--- NHẬN XÉT QUAN TRỌNG TỪ KẾT QUẢ ---")
print(f"1. Câu A và Câu B (cùng chuyên môn Backend, khác cách viết) : {sim_df.iloc[0, 1]*100:.1f}% -> Rất tương đồng")
print(f"2. Câu A và Câu C (cùng chuyên môn Backend, Tiếng Việt vs Anh): {sim_df.iloc[0, 2]*100:.1f}% -> Cực kỳ tương đồng")
print(f"3. Câu A và Câu D (Backend vs Hành chính nhân sự)             : {sim_df.iloc[0, 3]*100:.1f}% -> Gần như không liên quan")


=== MA TRẬN ĐỘ TƯƠNG ĐỒNG NGỮ NGHĨA (COSINE SIMILARITY) ===
                                    A. Python Backend (Tiếng Việt 1)  B. Django/FastAPI (Tiếng Việt 2)  C. Python Backend (Tiếng Anh)  D. Nhân sự hành chính (Khác ngành)
A. Python Backend (Tiếng Việt 1)                               1.000                             0.457                          0.706                               0.059
B. Django/FastAPI (Tiếng Việt 2)                               0.457                             1.000                          0.583                               0.225
C. Python Backend (Tiếng Anh)                                  0.706                             0.583                          1.000                               0.099
D. Nhân sự hành chính (Khác ngành)                             0.059                             0.225                          0.099                               1.000

--- NHẬN XÉT QUAN TRỌNG TỪ KẾT QUẢ ---
1. Câu A và Câu B (cùng chuyên môn Backend, khác c

---
### PHẦN 4: ÁP DỤNG MÃ HÓA TRÊN DỮ LIỆU TIN TUYỂN DỤNG THỰC TẾ (TẦNG SILVER)

Chúng ta kết nối dữ liệu tin việc làm thực tế từ dự án:
1. Kết hợp các trường văn bản: `title_normalized` + `requirements_clean` + `description_clean`.
2. Lấy mẫu 30 tin tuyển dụng thực tế thuộc nhiều chuyên môn:
   - Frontend Developer (React, Vue)
   - Backend Developer (Java, Python, Golang)
   - Data / AI Engineer (SQL, Machine Learning, Spark)
   - Mobile Developer (Flutter, iOS)
   - DevOps / Cloud Engineer (Docker, AWS)
3. Tạo ma trận `job_embeddings`.


In [4]:
# Nạp dữ liệu thực tế từ extractor của dự án
# Tự động tìm thư mục gốc dự án (chứa thư mục 'src')
def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'src')):
            return curr
        curr = os.path.dirname(curr)
    return os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.pipelines.silver.extractor import load_bronze_data
from src.pipelines.silver.transformers.jobs import clean_job_title
from src.pipelines.silver.transformers.job_descriptions import clean_html_text

bronze_data = load_bronze_data()
df_jobs = bronze_data["df_merged"].copy()

# Chọn mẫu 30 việc làm đa dạng ngành nghề
sample_jobs = df_jobs.head(30).copy()

job_texts = []
job_metas = []

for idx, r in sample_jobs.iterrows():
    title_clean = clean_job_title(str(r.get('title_raw', '')))
    req_clean = clean_html_text(str(r.get('requirements_raw', '')))[:300]
    desc_clean = clean_html_text(str(r.get('description_raw', '')))[:200]
    
    # Kết hợp văn bản đại diện cho công việc
    combined_repr = f"Vị trí: {title_clean}. Yêu cầu: {req_clean}. Mô tả: {desc_clean}"
    job_texts.append(combined_repr)
    
    job_metas.append({
        "job_id": r["job_id"],
        "title": title_clean,
        "company": str(r.get("company_name_raw", "Khác")),
        "city": str(r.get("location_raw", "Khác"))
    })

# Mã hóa ma trận 30 việc làm thành vector
print(f"Đang mã hóa {len(job_texts)} việc làm thực tế...")
job_embeddings = model.encode(job_texts, show_progress_bar=False)

print(f"Đã mã hóa thành công ma trận vector việc làm: {job_embeddings.shape}")
print(f"(Gồm 30 hàng, mỗi hàng là một vector 384 chiều đại diện cho 1 việc làm)")


-> Nạp dữ liệu nguồn Bronze từ đĩa...
   Đã nạp thành công 2,752 bản ghi đồng bộ vào bộ nhớ.
Đang mã hóa 30 việc làm thực tế...
Đã mã hóa thành công ma trận vector việc làm: (30, 384)
(Gồm 30 hàng, mỗi hàng là một vector 384 chiều đại diện cho 1 việc làm)


---
### PHẦN 5: THỬ NGHIỆM GỢI Ý VIỆC LÀM (CONTENT-BASED MATCHING DEMO)

Bây giờ chúng ta đóng vai trò là một ứng viên nhập hồ sơ / nguyện vọng tìm việc:
- Ứng viên gõ một câu tự do:
  *Ví dụ 1*: `"Tôi có kinh nghiệm làm việc với dữ liệu, thành thạo viết truy vấn SQL và trực quan hóa báo cáo Power BI"`
  *Ví dụ 2*: `"Lập trình viên muốn làm Frontend với ReactJS, Next.js và TypeScript"`
- Hệ thống:
  1. Mã hóa câu của ứng viên thành vector `query_vector`.
  2. Tính Cosine Similarity với 30 vector việc làm trong kho.
  3. Xếp hạng và trả về Top 3 việc làm phù hợp nhất kèm điểm số tương đồng.


In [5]:
def search_matching_jobs(candidate_query: str, top_k: int = 3):
    print(f"\n=======================================================")
    print(f"HỒ SƠ / NGUYỆN VỌNG ỨNG VIÊN:")
    print(f"'{candidate_query}'")
    print(f"=======================================================")
    
    # 1. Mã hóa truy vấn ứng viên thành vector 384 chiều
    candidate_vec = model.encode([candidate_query])
    
    # 2. Tính toán độ tương đồng Cosine giữa ứng viên và tất cả việc làm
    scores = cosine_similarity(candidate_vec, job_embeddings)[0]
    
    # 3. Lấy top K việc làm có điểm tương đồng cao nhất
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        meta = job_metas[idx]
        score_pct = round(scores[idx] * 100, 1)
        results.append({
            "Hạng": rank,
            "Độ phù hợp (Match)": f"{score_pct}%",
            "Chức danh việc làm": meta["title"],
            "Công ty": meta["company"],
            "Địa điểm": meta["city"]
        })
    
    df_res = pd.DataFrame(results)
    print(df_res.to_string(index=False))
    return df_res

# Thử nghiệm 1: Ứng viên chuyên về Data / SQL / BI
q1 = "Tôi có kinh nghiệm làm việc với dữ liệu, thành thạo viết truy vấn SQL và trực quan hóa báo cáo Power BI"
res1 = search_matching_jobs(q1, top_k=3)

# Thử nghiệm 2: Ứng viên Frontend ReactJS / TypeScript
q2 = "Lập trình viên giao diện Frontend có kinh nghiệm với React, TypeScript và thiết kế Figma"
res2 = search_matching_jobs(q2, top_k=3)



HỒ SƠ / NGUYỆN VỌNG ỨNG VIÊN:
'Tôi có kinh nghiệm làm việc với dữ liệu, thành thạo viết truy vấn SQL và trực quan hóa báo cáo Power BI'
 Hạng  Độ phù hợp (Match)                                                 Chức danh việc làm                                          Công ty                                                  Địa điểm
    1               60.0%                               Chuyên Viên Phân Tích Dữ Liệu Bán Lẻ                CÔNG TY CỔ PHẦN THỜI TRANG SIXDO. Lô 5, KCN Vĩnh Tuy, Đ. Lĩnh Nam, Phường Hoàng Mai, Hà Nội
    2 54.599998474121094% Data Analyst - Vận Hành Xanh Bike - Làm Việc Tại Royal City Hà Nội CÔNG TY CỔ PHẦN DI CHUYỂN XANH VÀ THÔNG MINH GSM    Tòa Royal City, Nguyễn Trãi, Phường Thanh Xuân, Hà Nội
    3  53.70000076293945%            Business Analyst - Data Hệ Thống Phân Phối - Chế Độ Tốt     CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL                                   Phường Cầu Giấy, Hà Nội

HỒ SƠ / NGUYỆN VỌNG ỨNG VIÊN:
'Lập trình viên giao diện Fronte

---
### PHẦN 6: ĐÁNH GIÁ HIỆU NĂNG & KẾ HOẠCH TRIỂN KHAI CHO 2,752 TIN

#### 1. Đánh giá tài nguyên & tốc độ:
- **Tốc độ mã hóa (Inference Speed)**: Khoảng **40 - 70 bài đăng / giây** trên CPU thông thường.
  -> Toàn bộ 2,752 tin việc làm chỉ mất khoảng **40 - 60 giây** để tạo xong toàn bộ vector.
- **Dung lượng lưu trữ (Storage Footprint)**:
  - 2,752 tin việc làm $\times$ 384 số thực (float32 = 4 bytes) = **~4.22 MB**.
  - Đây là kích thước rất nhỏ gọn, có thể lưu vào một tệp `data/embeddings/jobs_embeddings.npy` hoặc lưu vào cột vector trong PostgreSQL.
  - Khi ứng dụng Streamlit khởi chạy, việc nạp 4.2 MB vector vào RAM chỉ mất chưa đầy **0.05 giây**.

#### 2. Kế hoạch đưa vào sản phẩm chính thức:
1. **Bước tiếp theo (Phần 3 chính thức)**:
   - Viết module `src/embeddings/generator.py` để tự động hóa việc tính toán vector cho toàn bộ 2,752 tin và lưu vào `data/embeddings/`.
2. **Bước hoàn thiện (Phần 4 chính thức)**:
   - Viết module `src/recommender/matcher.py` tính toán kết hợp **Semantic Cosine Similarity** + **Skill Overlap (Jaccard Score)**.
   - Thêm tính năng **Gợi Ý Việc Làm Cá Nhân Hóa** trên giao diện Web Streamlit.


---
### PHẦN 7: KHAI PHÁ & PHÂN TÍCH PHÂN BỐ TOÀN BỘ KHÔNG GIAN VECTOR THỰC TẾ (3,249 TIN VIỆC LÀM)

Sau khi thử nghiệm trên tập mẫu nhỏ, chúng ta tiến hành **phân tích chuyên sâu trên toàn bộ 3,249 vector đặc trưng (384 chiều)** được tạo ra từ pipeline sản xuất của dự án (`data/embeddings/jobs_embeddings.npy` và `data/embeddings/jobs_metadata.json`).

Mục tiêu phân tích:
1. **Kiểm tra chuẩn hóa L2 và phân bố độ dài văn bản** giữa các nguồn crawl (TopCV, VietnamWorks, TopDev).
2. **Khảo sát phân bố độ tương đồng Cosine (Pairwise Cosine Similarity Distribution)** trong toàn bộ không gian tuyển dụng IT.
3. **Phân tích ma trận tương đồng ngữ nghĩa giữa các Domain / Career Track (Cross-Track Semantic Distance)** với Centroid Vector.
4. **Chiếu giảm chiều PCA (2D Projection)** để trực quan hóa bản đồ phân cụm việc làm thị trường IT.
5. **Khai phá các cặp công việc đặc biệt (Extreme Pairs Analysis)**: Cặp tin trùng lặp cao nhất và cặp tin xa nhau nhất.

In [ ]:
# 1. Nạp toàn bộ 3,249 vector embeddings & metadata từ dữ liệu sản xuất
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập đường dẫn tới data/embeddings
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(PROJECT_ROOT, 'data')):
    # Fallback nếu đang chạy từ root
    PROJECT_ROOT = os.path.abspath(os.getcwd())

EMB_PATH = os.path.join(PROJECT_ROOT, "data", "embeddings", "jobs_embeddings.npy")
META_PATH = os.path.join(PROJECT_ROOT, "data", "embeddings", "jobs_metadata.json")

embeddings = np.load(EMB_PATH)
with open(META_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f" Đã nạp thành công ma trận Embeddings: {embeddings.shape} ({embeddings.shape[0]} việc làm, {embeddings.shape[1]} chiều)")
print(f" Số lượng metadata tương ứng: {len(metadata)}")

# Kiểm tra L2 Norm của các vector
norms = np.linalg.norm(embeddings, axis=1)
print(f" L2 Norm trung bình: {norms.mean():.4f} (Độ lệch chuẩn: {norms.std():.6f})")

# Phân tích độ dài văn bản mô tả theo từng nguồn crawl
df_meta = pd.DataFrame(metadata)
df_meta['text_len'] = df_meta['combined_text'].str.len()
df_meta['word_count'] = df_meta['combined_text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df_meta, x='source', y='text_len', palette='Blues', ax=axes[0])
axes[0].set_title("Phân Bố Độ Dài Ký Tự (Text Length) Theo Nguồn Tuyển Dụng", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Số lượng ký tự")

sns.histplot(data=df_meta, x='word_count', hue='source', kde=True, bins=40, palette='Set2', ax=axes[1])
axes[1].set_title("Phân Bố Số Lượng Từ (Word Count) Của Tin Tuyển Dụng", fontsize=11, fontweight='bold')
axes[1].set_xlabel("Số từ")
plt.tight_layout()
plt.show()

print("
Thống kê độ dài văn bản theo sàn:")
print(df_meta.groupby('source')['word_count'].describe()[['count', 'mean', 'std', 'min', '50%', 'max']])

---
#### 2. Phân Bố Độ Tương Đồng Cosine (Pairwise Cosine Similarity Distribution)

Một vấn đề phổ biến trong không gian vector đa chiều (như 384d của MiniLM) là **Hubness Problem** hoặc **Semantic Congestion** (các vector bị co cụm lại, khiến độ tương đồng trung bình quá cao hoặc quá thấp). 

Đoạn code dưới đây lấy mẫu ngẫu nhiên 1,000 công việc (tương đương $\approx 500,000$ cặp khoảng cách) để tính toán hàm mật độ xác suất của Cosine Similarity.

In [ ]:
# 2. Tính toán phân bố Pairwise Cosine Similarity trên 1,000 tin tuyển dụng ngẫu nhiên
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
sample_indices = np.random.choice(len(embeddings), size=min(1000, len(embeddings)), replace=False)
sample_embs = embeddings[sample_indices]

# Tính ma trận tương đồng Cosine
cos_sim_matrix = cosine_similarity(sample_embs)
# Lấy nửa trên của ma trận (bỏ đường chéo chính)
triu_indices = np.triu_indices(len(sample_embs), k=1)
pairwise_similarities = cos_sim_matrix[triu_indices]

# Tính các chỉ số thống kê
sim_mean = np.mean(pairwise_similarities)
sim_std = np.std(pairwise_similarities)
sim_median = np.median(pairwise_similarities)
p5, p95 = np.percentile(pairwise_similarities, [5, 95])

print("=" * 60)
print("THỐNG KÊ PHÂN BỐ ĐỘ TƯƠNG ĐỒNG COSINE (N = 499,500 cặp)")
print("=" * 60)
print(f"• Mean (Trung bình):      {sim_mean:.4f}")
print(f"• Median (Trung vị):       {sim_median:.4f}")
print(f"• Standard Deviation:     {sim_std:.4f}")
print(f"• 5th Percentile (P5):    {p5:.4f}")
print(f"• 95th Percentile (P95):  {p95:.4f}")
print(f"• Min - Max:              {pairwise_similarities.min():.4f} - {pairwise_similarities.max():.4f}")

# Vẽ biểu đồ phân phối
plt.figure(figsize=(10, 4.5))
sns.histplot(pairwise_similarities, bins=60, kde=True, color='#2563EB', stat="density")
plt.axvline(sim_mean, color='red', linestyle='--', linewidth=1.5, label=f'Mean = {sim_mean:.3f}')
plt.axvline(sim_median, color='orange', linestyle=':', linewidth=1.5, label=f'Median = {sim_median:.3f}')
plt.axvline(p5, color='gray', linestyle='-.', alpha=0.7, label=f'P5={p5:.2f}, P95={p95:.2f}')
plt.axvline(p95, color='gray', linestyle='-.', alpha=0.7)

plt.title("Phân Bố Xác Suất Của Độ Tương Đồng Cosine Giữa Các Tin Tuyển Dụng IT", fontsize=12, fontweight='bold')
plt.xlabel("Cosine Similarity")
plt.ylabel("Mật độ xác suất (Density)")
plt.legend(frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
#### 3. Ma Trận Khoảng Cách Ngữ Nghĩa Giữa Các Career Track / Domain (Cross-Track Centroid Heatmap)

Thị trường IT có nhiều chuyên môn khác nhau: Backend, Frontend, Data Engineer/Analyst, AI/ML, DevOps, Mobile, QA/QC.
Chúng ta gom nhóm các bài tuyển dụng theo chuyên môn, tính **Vector trọng tâm (Centroid Vector)** của từng chuyên môn trong không gian 384 chiều:
$$\vec{c}_{track} = \frac{1}{N} \sum_{i \in track} \vec{v}_i$$
Sau đó tính toán mức độ gần gũi ngữ nghĩa giữa các lĩnh vực.

In [ ]:
# 3. Phân tích tương quan ngữ nghĩa giữa các Domain / Career Track
# Nhận diện chuyên môn bài đăng dựa vào title và kỹ năng
def assign_track(title, skills):
    t = str(title).lower()
    s = [str(x).lower() for x in (skills or [])]
    
    if any(k in t for k in ['ai', 'machine learning', 'deep learning', 'nlp', 'computer vision', 'data science']):
        return 'AI / Machine Learning'
    if any(k in t for k in ['data engineer', 'data analyst', 'bi developer', 'big data', 'etl', 'sql']) or any(x in s for x in ['power bi', 'tableau', 'spark', 'hadoop']):
        return 'Data / Business Intelligence'
    if any(k in t for k in ['devops', 'cloud', 'aws', 'sre', 'system', 'kubernetes', 'infrastructure']):
        return 'DevOps / Cloud'
    if any(k in t for k in ['mobile', 'android', 'ios', 'flutter', 'react native']):
        return 'Mobile App Dev'
    if any(k in t for k in ['qa', 'qc', 'tester', 'test', 'kiem thu', 'automation test']):
        return 'QA / Software Testing'
    if any(k in t for k in ['frontend', 'front end', 'react', 'vue', 'angular']):
        return 'Frontend Dev'
    if any(k in t for k in ['backend', 'back end', 'java', 'golang', 'c#', '.net', 'php', 'python', 'node']):
        return 'Backend Dev'
    return 'Other / General Software'

# Phân loại track cho 3,249 việc làm
tracks = [assign_track(m.get('title', ''), m.get('skills', [])) for m in metadata]
df_meta['track'] = tracks

# Lấy các track chính (bỏ Other)
major_tracks = [
    'Backend Dev', 'Frontend Dev', 'Mobile App Dev', 
    'DevOps / Cloud', 'Data / Business Intelligence', 
    'AI / Machine Learning', 'QA / Software Testing'
]

centroids = {}
for tr in major_tracks:
    idxs = [i for i, t in enumerate(tracks) if t == tr]
    if idxs:
        # Vector trung bình chuẩn hóa L2
        c_vec = np.mean(embeddings[idxs], axis=0)
        c_vec = c_vec / np.linalg.norm(c_vec)
        centroids[tr] = c_vec
        print(f"• Track '{tr}': {len(idxs)} việc làm")

# Ma trận tương đồng giữa các Centroid
track_names = list(centroids.keys())
c_matrix = np.array([centroids[tr] for tr in track_names])
sim_matrix = cosine_similarity(c_matrix)

# Vẽ Heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(sim_matrix, annot=True, fmt=".3f", cmap="YlGnBu",
            xticklabels=track_names, yticklabels=track_names,
            cbar_kws={'label': 'Cosine Similarity'})
plt.title("Ma Trận Tương Đồng Ngữ Nghĩa Giữa Các Nhóm Nghề IT (Cross-Track Centroid)", fontsize=11, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
#### 4. Chiếu Giảm Chiều Không Gian Ngữ Nghĩa Về 2D (PCA Projection)

Để có cái nhìn trực quan về sự phân bố và phân cụm của toàn bộ 3,249 tin việc làm, chúng ta dùng **Principal Component Analysis (PCA)** để nén từ 384 chiều về 2 chiều đại diện lớn nhất.

In [ ]:
# 4. Trực quan hóa 2D không gian ngữ nghĩa bằng PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
embs_2d = pca.fit_transform(embeddings)

print(f"Tỷ lệ phương sai giải thích được của 2 thành phần chính: {pca.explained_variance_ratio_ * 100}%")
print(f"Tổng phương sai lưu giữ trên 2D: {np.sum(pca.explained_variance_ratio_) * 100:.2f}%")

df_plot = pd.DataFrame({
    'PCA1': embs_2d[:, 0],
    'PCA2': embs_2d[:, 1],
    'Track': df_meta['track']
})

# Lọc chỉ hiển thị các track chính
df_plot_filtered = df_plot[df_plot['Track'].isin(major_tracks)]

plt.figure(figsize=(12, 8))
palette = sns.color_palette("tab10", len(major_tracks))
sns.scatterplot(
    data=df_plot_filtered,
    x='PCA1', y='PCA2',
    hue='Track',
    palette=palette,
    alpha=0.6,
    s=35,
    edgecolor='none'
)

# Đánh dấu tâm cụm của từng nhóm
for tr in major_tracks:
    sub = df_plot[df_plot['Track'] == tr]
    if len(sub) > 0:
        plt.scatter(sub['PCA1'].mean(), sub['PCA2'].mean(), color='black', marker='X', s=120)
        plt.annotate(
            tr, 
            (sub['PCA1'].mean(), sub['PCA2'].mean()),
            textcoords="offset points", 
            xytext=(0, 7), 
            ha='center', 
            fontsize=9, 
            fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.2", fc="yellow", alpha=0.7)
        )

plt.title("Bản Đồ Không Gian Ngữ Nghĩa Của 3,249 Việc Làm IT (PCA 2D Projection)", fontsize=13, fontweight='bold')
plt.xlabel(f"PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
#### 5. Khai Phá Các Cặp Tin Tuyển Dụng Đặc Biệt (Extreme Pairs Analysis)

Một ứng dụng giá trị của Semantic Embedding là:
- **Phát hiện trùng lặp (Near-Duplicate Detection)**: Tìm các bài tuyển dụng có Cosine Similarity $\ge 0.95$ (do cùng 1 công ty đăng lên nhiều sàn TopCV, VietnamWorks, TopDev với tiêu đề hơi khác nhau).
- **Phát hiện đối nghịch ngữ nghĩa**: Tìm các vị trí công việc có khoảng cách xa nhất trong ngành IT.

In [ ]:
# 5. Khai phá các cặp việc làm đặc biệt (Highest vs Lowest Similarity)
# Lấy mẫu 800 tin để tìm kiếm nhanh các cặp cực trị
np.random.seed(123)
sub_idx = np.random.choice(len(embeddings), size=min(800, len(embeddings)), replace=False)
sub_embs = embeddings[sub_idx]
sub_sim = cosine_similarity(sub_embs)

# Nửa trên ma trận
r_idx, c_idx = np.triu_indices(len(sub_embs), k=1)
sim_values = sub_sim[r_idx, c_idx]

# Sắp xếp chỉ số
sorted_args = np.argsort(sim_values)

print("=" * 80)
print("TOP 3 CẶP VIỆC LÀM TRÙNG LẶP / TƯƠNG ĐỒNG CAO NHẤT (NEAR-DUPLICATES)")
print("=" * 80)
for i in range(1, 4):
    pair_pos = sorted_args[-i]
    idx1 = sub_idx[r_idx[pair_pos]]
    idx2 = sub_idx[c_idx[pair_pos]]
    score = sim_values[pair_pos]
    m1, m2 = metadata[idx1], metadata[idx2]
    print(f"
[Cặp {i} - Cosine: {score:.4f}]")
    print(f"  Tin A ({m1.get('source')}): [{m1.get('company')}] {m1.get('title')}")
    print(f"  Tin B ({m2.get('source')}): [{m2.get('company')}] {m2.get('title')}")

print("
" + "=" * 80)
print("TOP 3 CẶP VIỆC LÀM KHÁC BIỆT NHẤT VỀ MẶT NGỮ NGHĨA")
print("=" * 80)
for i in range(3):
    pair_pos = sorted_args[i]
    idx1 = sub_idx[r_idx[pair_pos]]
    idx2 = sub_idx[c_idx[pair_pos]]
    score = sim_values[pair_pos]
    m1, m2 = metadata[idx1], metadata[idx2]
    print(f"
[Cặp {i+1} - Cosine: {score:.4f}]")
    print(f"  Tin A ({m1.get('source')}): [{m1.get('company')}] {m1.get('title')}")
    print(f"  Tin B ({m2.get('source')}): [{m2.get('company')}] {m2.get('title')}")